# Analyse exploratoire des données électorales

Ce notebook a pour objectif de **vérifier les hypothèses de travail** utilisées dans `utils.py` et de montrer une **compréhension explicite** du jeu de données avant de répondre aux questions dans `main.ipynb`.

On se concentre ici sur :
- la structure générale du jeu de données ;
- les types et valeurs manquantes ;
- les doublons ;
- la structure des codes administratifs ;
- l'identification des lignes spéciales (`abstentions`, `blancs`, `nuls`).


In [ ]:
import pandas as pd
from utils import load_data, build_code_commune, build_candidat

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 30)

df = load_data()


## 1. Dimensions et aperçu général

On commence par vérifier la taille du jeu de données, les types de colonnes, puis un aperçu rapide avec `head()` et `sample()`.


In [ ]:
print("Dimensions :", df.shape)
print("\nTypes :")
print(df.dtypes)

print("\nPremières lignes :")
display(df.head())

print("\nÉchantillon aléatoire :")
display(df.sample(10, random_state=42))


Dimensions : (528675, 7)

Types :
code_departement       object
libelle_departement    object
code_commune           object
libelle_commune        object
prenom                 object
nom                    object
voix                    int64
dtype: object

Premières lignes :


,code_departement,libelle_departement,code_commune,libelle_commune,prenom,nom,voix
0,01,Ain,001,L'Abergement-Clémenciat,Nathalie,ARTHAUD,3
1,01,Ain,002,L'Abergement-de-Varey,Nathalie,ARTHAUD,2
2,01,Ain,004,Ambérieu-en-Bugey,Nathalie,ARTHAUD,38
3,01,Ain,005,Ambérieux-en-Dombes,Nathalie,ARTHAUD,8
4,01,Ain,006,Ambléon,Nathalie,ARTHAUD,0



Échantillon aléatoire :


,code_departement,libelle_departement,code_commune,libelle_commune,prenom,nom,voix
201455,64,Pyrénées-Atlantiques,386,Mirepeix,Éric,ZEMMOUR,59
359991,21,Côte-d'Or,618,Talmay,Philippe,POUTOU,3
414755,69,Rhône,091,Givors,Nicolas,DUPONT-AIGNAN,100
21188,57,Moselle,459,Merschweiller,Nathalie,ARTHAUD,2
77268,2B,Haute-Corse,134,L'Ile-Rousse,Emmanuel,MACRON,276
518561,64,Pyrénées-Atlantiques,285,Juxue,NaN,nuls,0
24091,62,Pas-de-Calais,605,Neulette,Nathalie,ARTHAUD,0
339320,59,Nord,335,Lecelles,Valérie,PÉCRESSE,65
362524,27,Eure,468,Pont-Authou,Philippe,POUTOU,5
376225,62,Pas-de-Calais,280,Dury,Philippe,POUTOU,1


## 2. Valeurs manquantes

Cette étape permet d'identifier les colonnes incomplètes et de vérifier si les valeurs manquantes ont une signification métier.


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing


prenom                 105735
libelle_departement         0
code_departement            0
code_commune                0
libelle_commune             0
nom                         0
voix                        0
dtype: int64

In [ ]:
df[df["prenom"].isna()]["nom"].value_counts(dropna=False)


nom
abstentions    35245
blancs         35245
nuls           35245
Name: count, dtype: int64

## 3. Vérification des doublons

Le sujet insiste sur les bonnes pratiques : on vérifie donc l'absence de doublons exacts et de doublons sur une clé descriptive raisonnable.


In [ ]:
cles = [
    "code_departement",
    "libelle_departement",
    "code_commune",
    "libelle_commune",
    "prenom",
    "nom",
]

print("Doublons exacts :", df.duplicated().sum())
print("Doublons sur la clé descriptive :", df.duplicated(subset=cles).sum())


Doublons exacts : 0
Doublons sur la clé descriptive : 0


## 4. Vérification des codes administratifs

Cette partie sert à justifier les choix faits dans `utils.py` :
- `code_commune` doit être lu comme une chaîne car il contient des zéros initiaux ;
- `code_departement` est déjà correctement formaté dans les données chargées ;
- les lignes `fr_etranger` doivent être traitées à part.


In [ ]:
print("Exemples de code_commune :")
print(df["code_commune"].head(20).tolist())

print("\nLongueur des code_commune :")
print(df["code_commune"].astype(str).str.len().value_counts().sort_index())

print("\nQuelques code_commune commençant par 0 :")
display(
    df[df["code_commune"].astype(str).str.startswith("0")][
        ["code_departement", "code_commune"]
    ].head(20)
)


Exemples de code_commune :
['001', '002', '004', '005', '006', '007', '008', '009', '010', '011', '012', '013', '014', '015', '016', '017', '019', '021', '022', '023']

Longueur des code_commune :
code_commune
3    528675
Name: count, dtype: int64

Quelques code_commune commençant par 0 :


,code_departement,code_commune
0,01,001
1,01,002
2,01,004
3,01,005
4,01,006
5,01,007
6,01,008
7,01,009
8,01,010
9,01,011


In [ ]:
codes_dept = sorted(df["code_departement"].dropna().astype(str).unique())

print("Premiers codes département :")
print(codes_dept[:30])

print("\nDerniers codes département :")
print(codes_dept[-30:])

print("\nLongueur des code_departement :")
print(df["code_departement"].astype(str).str.len().value_counts().sort_index())

print("\nLignes fr_etranger :")
display(
    df[df["code_departement"].astype(str).str.contains("fr_", na=False)][
        ["code_departement", "libelle_departement"]
    ].drop_duplicates()
)


Premiers codes département :
['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '21', '22', '23', '24', '25', '26', '27', '28', '29', '2A', '2B']

Derniers codes département :
['78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '971', '972', '973', '974', '975', '976', '977', '978', '986', '987', '988', 'fr_etranger']

Longueur des code_departement :
code_departement
2     522300
3       3225
11      3150
Name: count, dtype: int64

Lignes fr_etranger :


,code_departement,libelle_departement
35035,fr_etranger,Français établis hors de France


## 5. Vérification des transformations métier

On teste ici les fonctions de `utils.py` pour vérifier qu'elles correspondent bien à la structure observée des données.


In [ ]:
df_code = build_code_commune(df)

print("Aperçu après build_code_commune :")
display(df_code[["code_departement", "code_commune", "libelle_commune"]].head(20))

print("Longueur des code_commune après reconstruction hors fr_etranger :")
print(
    df_code.loc[
        ~df_code["code_departement"].astype(str).str.startswith("fr_"),
        "code_commune",
    ]
    .astype(str)
    .str.len()
    .value_counts()
    .sort_index()
)

print("\nExemples fr_etranger après transformation :")
display(
    df_code[df_code["code_departement"].astype(str).str.startswith("fr_")][
        ["code_departement", "code_commune", "libelle_commune"]
    ].head(20)
)


Aperçu après build_code_commune :


,code_departement,code_commune,libelle_commune
0,01,01001,L'Abergement-Clémenciat
1,01,01002,L'Abergement-de-Varey
2,01,01004,Ambérieu-en-Bugey
3,01,01005,Ambérieux-en-Dombes
4,01,01006,Ambléon
5,01,01007,Ambronay
6,01,01008,Ambutrix
7,01,01009,Andert-et-Condon
8,01,01010,Anglefort
9,01,01011,Apremont


Longueur des code_commune après reconstruction hors fr_etranger :
code_commune
5    522300
6      3225
Name: count, dtype: int64

Exemples fr_etranger après transformation :


,code_departement,code_commune,libelle_commune
35035,fr_etranger,001,Abidjan
35036,fr_etranger,002,Abou Dabi
35037,fr_etranger,003,Abuja
35038,fr_etranger,004,Accra
35039,fr_etranger,006,Addis Abeba
35040,fr_etranger,007,Agadir
35041,fr_etranger,008,Alexandrie
35042,fr_etranger,009,Alger
35043,fr_etranger,010,Almaty
35044,fr_etranger,011,Amman


In [ ]:
df_cand = build_candidat(df)

print("Nombre de candidat manquants :", df_cand["candidat"].isna().sum())

print("\nLignes où candidat est manquant :")
display(df_cand[df_cand["candidat"].isna()][["prenom", "nom"]].drop_duplicates())

print("\nExemples de candidats construits :")
print(df_cand["candidat"].dropna().sample(20, random_state=42).tolist())


Nombre de candidat manquants : 105735

Lignes où candidat est manquant :


,prenom,nom
422940,NaN,abstentions
458185,NaN,blancs
493430,NaN,nuls



Exemples de candidats construits :
['Nicolas DUPONT-AIGNAN', 'Philippe POUTOU', 'Philippe POUTOU', 'Valérie PÉCRESSE', 'Nathalie ARTHAUD', 'Yannick JADOT', 'Nicolas DUPONT-AIGNAN', 'Yannick JADOT', 'Yannick JADOT', 'Anne HIDALGO', 'Fabien ROUSSEL', 'Éric ZEMMOUR', 'Marine LE PEN', 'Yannick JADOT', 'Nathalie ARTHAUD', 'Anne HIDALGO', 'Jean LASSALLE', 'Nicolas DUPONT-AIGNAN', 'Jean LASSALLE', 'Yannick JADOT']


## 6. Conclusion

Les vérifications précédentes permettent de justifier les hypothèses principales utilisées dans `utils.py` :

- `code_commune` doit être traité comme un identifiant texte ;
- `code_departement` est déjà bien formaté dans les données chargées ;
- les lignes spéciales (`abstentions`, `blancs`, `nuls`) sont repérées par `prenom` manquant ;
- les données ne présentent pas de doublons évidents ;
- les transformations de `utils.py` sont cohérentes avec la structure du jeu de données.

Ce notebook complète donc `main.ipynb` en montrant la **lecture et la compréhension initiales des données** avant l'analyse demandée dans le sujet.
